# Garmin Connect Dashboard
Interactive charts for your Garmin health and fitness data.
Run cells top-to-bottom. Credentials are entered securely with `getpass` and never stored.


## 1 · Login to Garmin Connect

In [ ]:
import os, json, warnings
from datetime import date, timedelta
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from garminconnect import Garmin
from dotenv import load_dotenv

warnings.filterwarnings('ignore')

# Load credentials from .env file sitting next to this notebook
load_dotenv(Path(__file__).parent / '.env' if '__file__' in dir() else Path('.env'))
email    = os.getenv('GARMIN_EMAIL')
password = os.getenv('GARMIN_PASSWORD')

if not email or not password:
    import getpass
    email    = input("Garmin email: ")
    password = getpass.getpass("Garmin password: ")

api = Garmin(email, password)
api.login()
print("Logged in as:", email)

TODAY = date.today()


## 2 · Steps — Last 30 Days

In [ ]:
rows = []
for i in range(30):
    d = (TODAY - timedelta(days=i)).isoformat()
    try:
        data = api.get_steps_data(d)
        total = sum(s.get('steps', 0) for s in (data or []))
        rows.append({'date': d, 'steps': total})
    except Exception:
        pass

df_steps = pd.DataFrame(rows).sort_values('date')

fig = go.Figure()
fig.add_trace(go.Bar(x=df_steps['date'], y=df_steps['steps'],
    marker_color='#27ae60', name='Steps'))
fig.add_hline(y=10000, line_dash='dash', line_color='red',
    annotation_text='10,000 goal', annotation_position='top right')
fig.update_layout(title='Daily Steps — Last 30 Days',
    xaxis_title='Date', yaxis_title='Steps',
    template='plotly_white', height=400)
fig.show()

avg = df_steps['steps'].mean()
goal_days = (df_steps['steps'] >= 10000).sum()
print(f"30-day average : {avg:,.0f} steps/day")
print(f"Days >= 10,000 : {goal_days}/30")


## 3 · Heart Rate — Last 30 Days

In [ ]:
hr_rows = []
for i in range(30):
    d = (TODAY - timedelta(days=i)).isoformat()
    try:
        data = api.get_heart_rates(d)
        if data:
            hr_rows.append({
                'date':       d,
                'resting_hr': data.get('restingHeartRate'),
                'max_hr':     data.get('maxHeartRate'),
                'min_hr':     data.get('minHeartRate'),
            })
    except Exception:
        pass

df_hr = pd.DataFrame(hr_rows).sort_values('date').dropna(subset=['resting_hr'])

fig = go.Figure()
fig.add_trace(go.Scatter(x=df_hr['date'], y=df_hr['max_hr'],
    mode='lines', name='Max HR',
    line=dict(color='#e74c3c', width=1, dash='dot')))
fig.add_trace(go.Scatter(x=df_hr['date'], y=df_hr['resting_hr'],
    mode='lines+markers', name='Resting HR',
    line=dict(color='#8e44ad', width=2), marker=dict(size=6)))
fig.add_trace(go.Scatter(x=df_hr['date'], y=df_hr['min_hr'],
    mode='lines', name='Min HR',
    line=dict(color='#3498db', width=1, dash='dot')))
fig.update_layout(title='Heart Rate — Last 30 Days',
    xaxis_title='Date', yaxis_title='BPM',
    template='plotly_white', height=400)
fig.show()

if not df_hr.empty:
    print(f"Avg resting HR : {df_hr['resting_hr'].mean():.1f} bpm")
    print(f"Avg max HR     : {df_hr['max_hr'].mean():.1f} bpm")


## 4 · HRV Status — Last 30 Days

In [ ]:
hrv_rows = []
for i in range(30):
    d = (TODAY - timedelta(days=i)).isoformat()
    try:
        data = api.get_hrv_data(d)
        if data and data.get('hrvSummary'):
            s = data['hrvSummary']
            hrv_rows.append({
                'date':        d,
                'weekly_avg':  s.get('weeklyAvg'),
                'last_night':  s.get('lastNight'),
                'status':      s.get('status', ''),
            })
    except Exception:
        pass

df_hrv = pd.DataFrame(hrv_rows).sort_values('date')

if not df_hrv.empty:
    fig = go.Figure()
    fig.add_trace(go.Bar(x=df_hrv['date'], y=df_hrv['last_night'],
        name='Last Night', marker_color='rgba(26,188,156,0.7)'))
    fig.add_trace(go.Scatter(x=df_hrv['date'], y=df_hrv['weekly_avg'],
        mode='lines', name='7-day Avg',
        line=dict(color='#16a085', width=2.5)))
    fig.update_layout(title='HRV Status — Last 30 Days',
        xaxis_title='Date', yaxis_title='ms',
        template='plotly_white', height=400)
    fig.show()
    print(f"Avg last-night HRV : {df_hrv['last_night'].mean():.1f} ms")
    print(f"Avg weekly HRV     : {df_hrv['weekly_avg'].mean():.1f} ms")
else:
    print("No HRV data (requires a compatible Garmin device with wrist HRV tracking).")


## 5 · Sleep — Last 30 Days

In [ ]:
sleep_rows = []
for i in range(30):
    d = (TODAY - timedelta(days=i)).isoformat()
    try:
        data = api.get_sleep_data(d)
        if data and data.get('dailySleepDTO'):
            dto = data['dailySleepDTO']
            sleep_rows.append({
                'date':   d,
                'total':  (dto.get('sleepTimeSeconds') or 0) / 3600,
                'deep':   (dto.get('deepSleepSeconds') or 0) / 3600,
                'light':  (dto.get('lightSleepSeconds') or 0) / 3600,
                'rem':    (dto.get('remSleepSeconds') or 0) / 3600,
                'awake':  (dto.get('awakeSleepSeconds') or 0) / 3600,
                'score':  (dto.get('sleepScores') or {}).get('overall', {}).get('value'),
            })
    except Exception:
        pass

df_sleep = pd.DataFrame(sleep_rows).sort_values('date')
df_sleep = df_sleep[df_sleep['total'] > 0]

if not df_sleep.empty:
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
        subplot_titles=('Sleep Duration by Stage (hours)', 'Sleep Score'),
        row_heights=[0.65, 0.35])

    fig.add_trace(go.Bar(x=df_sleep['date'], y=df_sleep['deep'],
        name='Deep', marker_color='#1a237e'), 1, 1)
    fig.add_trace(go.Bar(x=df_sleep['date'], y=df_sleep['light'],
        name='Light', marker_color='#5c6bc0'), 1, 1)
    fig.add_trace(go.Bar(x=df_sleep['date'], y=df_sleep['rem'],
        name='REM', marker_color='#80cbc4'), 1, 1)
    fig.add_trace(go.Bar(x=df_sleep['date'], y=df_sleep['awake'],
        name='Awake', marker_color='#ef9a9a'), 1, 1)

    if df_sleep['score'].notna().any():
        fig.add_trace(go.Scatter(x=df_sleep['date'], y=df_sleep['score'],
            mode='lines+markers', name='Score',
            line=dict(color='#f39c12', width=2), marker=dict(size=6)), 2, 1)

    fig.update_layout(barmode='stack', template='plotly_white',
        height=600, title='Sleep — Last 30 Days')
    fig.show()

    print(f"Avg total sleep : {df_sleep['total'].mean():.2f} h/night")
    print(f"Avg deep sleep  : {df_sleep['deep'].mean():.2f} h")
    print(f"Avg REM sleep   : {df_sleep['rem'].mean():.2f} h")
    if df_sleep['score'].notna().any():
        print(f"Avg sleep score : {df_sleep['score'].mean():.0f}/100")
else:
    print("No sleep data returned.")


## 6 · Body Battery — Last 14 Days

In [ ]:
start_d = (TODAY - timedelta(days=14)).isoformat()
end_d   = TODAY.isoformat()

try:
    bb_data = api.get_body_battery(start_d, end_d)
    bb_rows = []
    for day in (bb_data or []):
        for entry in (day.get('bodyBatteryValuesArray') or []):
            if entry and len(entry) >= 2 and entry[1] is not None:
                bb_rows.append({
                    'time':    pd.to_datetime(entry[0], unit='ms'),
                    'battery': entry[1]
                })

    df_bb = pd.DataFrame(bb_rows).sort_values('time')

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df_bb['time'], y=df_bb['battery'],
        mode='lines', fill='tozeroy', name='Body Battery',
        line=dict(color='#f39c12', width=1.5),
        fillcolor='rgba(243,156,18,0.2)'))
    fig.add_hline(y=25, line_dash='dash', line_color='red',
        annotation_text='Low (25)', annotation_position='bottom right')
    fig.update_layout(title='Body Battery — Last 14 Days',
        xaxis_title='Time', yaxis_title='Level (0–100)',
        yaxis=dict(range=[0, 105]),
        template='plotly_white', height=400)
    fig.show()

    daily_max = df_bb.set_index('time').resample('D')['battery'].max()
    daily_min = df_bb.set_index('time').resample('D')['battery'].min()
    print(f"Avg daily max  : {daily_max.mean():.0f}")
    print(f"Avg daily min  : {daily_min.mean():.0f}")
except Exception as e:
    print(f"Body Battery error: {e}")


## 7 · Stress — Last 14 Days

In [ ]:
stress_rows = []
for i in range(14):
    d = (TODAY - timedelta(days=i)).isoformat()
    try:
        data = api.get_stress_data(d)
        if data:
            avg_stress = data.get('avgStressLevel')
            max_stress = data.get('maxStressLevel')
            stress_rows.append({'date': d, 'avg': avg_stress, 'max': max_stress})
    except Exception:
        pass

df_stress = pd.DataFrame(stress_rows).sort_values('date').dropna(subset=['avg'])

if not df_stress.empty:
    fig = go.Figure()
    fig.add_trace(go.Bar(x=df_stress['date'], y=df_stress['avg'],
        name='Avg Stress',
        marker=dict(
            color=df_stress['avg'],
            colorscale=[[0,'#2ecc71'],[0.4,'#f1c40f'],[0.7,'#e67e22'],[1,'#e74c3c']],
            cmin=0, cmax=100,
            showscale=True,
            colorbar=dict(title='Level')
        )))
    fig.update_layout(title='Daily Avg Stress — Last 14 Days',
        xaxis_title='Date', yaxis_title='Stress Level (0–100)',
        yaxis=dict(range=[0, 100]),
        template='plotly_white', height=400)
    fig.show()
    print(f"Avg stress level : {df_stress['avg'].mean():.1f}/100")
else:
    print("No stress data returned.")


## 8 · Recent Activities (last 50)

In [ ]:
try:
    activities = api.get_activities(0, 50)
    act_rows = []
    for a in (activities or []):
        act_rows.append({
            'date':         a.get('startTimeLocal','')[:10],
            'type':         a.get('activityType', {}).get('typeKey','').replace('_',' ').title(),
            'distance_km':  round((a.get('distance') or 0) / 1000, 2),
            'duration_min': round((a.get('duration') or 0) / 60, 1),
            'avg_hr':       a.get('averageHR'),
            'max_hr':       a.get('maxHR'),
            'calories':     a.get('calories'),
            'elevation_m':  a.get('elevationGain'),
        })

    df_act = pd.DataFrame(act_rows).sort_values('date', ascending=False)
    print(f"Showing {len(df_act)} recent activities:
")
    display(df_act[['date','type','distance_km','duration_min','avg_hr','calories','elevation_m']].head(20))

    # Pie: activity type breakdown
    type_counts = df_act['type'].value_counts()
    fig1 = px.pie(values=type_counts.values, names=type_counts.index,
        title='Activity Type Mix (last 50)',
        template='plotly_white',
        color_discrete_sequence=px.colors.qualitative.Set2)
    fig1.show()

    # Bar: monthly duration
    df_act['month'] = pd.to_datetime(df_act['date']).dt.to_period('M').astype(str)
    monthly = df_act.groupby('month')['duration_min'].sum().reset_index()
    fig2 = go.Figure(go.Bar(x=monthly['month'], y=monthly['duration_min'],
        marker_color='#9b59b6', name='Minutes'))
    fig2.update_layout(title='Monthly Activity Minutes',
        xaxis_title='Month', yaxis_title='Minutes',
        template='plotly_white', height=400)
    fig2.show()

except Exception as e:
    print(f"Activities error: {e}")


## 9 · Training Status & VO₂ Max

In [ ]:
# VO2 Max trend from activity data
vo2_rows = []
try:
    activities = api.get_activities(0, 100)
    for a in (activities or []):
        if a.get('vO2MaxValue'):
            vo2_rows.append({
                'date':  a.get('startTimeLocal','')[:10],
                'vo2':   a['vO2MaxValue'],
                'type':  a.get('activityType', {}).get('typeKey','')
            })
except Exception:
    pass

if vo2_rows:
    df_vo2 = pd.DataFrame(vo2_rows).sort_values('date')
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df_vo2['date'], y=df_vo2['vo2'],
        mode='lines+markers', name='VO₂ Max',
        line=dict(color='#e67e22', width=2), marker=dict(size=6)))
    fig.update_layout(title='VO₂ Max Over Time (from activities)',
        xaxis_title='Date', yaxis_title='mL/(kg·min)',
        template='plotly_white', height=400)
    fig.show()
    print(f"Latest VO₂ Max : {df_vo2['vo2'].iloc[-1]:.1f} mL/kg/min")
else:
    print("No VO₂ Max values in recent activities.")

# Training load
try:
    tl = api.get_training_load()
    if tl:
        print("\nTraining Load summary:")
        print(json.dumps(tl, indent=2)[:1500])
except Exception as e:
    print(f"Training load: {e}")


## 10 · Respiration Rate — Last 14 Days

In [ ]:
resp_rows = []
for i in range(14):
    d = (TODAY - timedelta(days=i)).isoformat()
    try:
        data = api.get_respiration_data(d)
        if data:
            avg_r = data.get('avgWakingRespirationValue') or data.get('avgSleepRespirationValue')
            resp_rows.append({'date': d, 'avg_brpm': avg_r,
                'sleep_brpm': data.get('avgSleepRespirationValue'),
                'wake_brpm':  data.get('avgWakingRespirationValue')})
    except Exception:
        pass

df_resp = pd.DataFrame(resp_rows).sort_values('date').dropna(subset=['avg_brpm'])

if not df_resp.empty:
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df_resp['date'], y=df_resp['wake_brpm'],
        mode='lines+markers', name='Awake',
        line=dict(color='#3498db', width=2), marker=dict(size=6)))
    fig.add_trace(go.Scatter(x=df_resp['date'], y=df_resp['sleep_brpm'],
        mode='lines+markers', name='Asleep',
        line=dict(color='#9b59b6', width=2), marker=dict(size=6)))
    fig.update_layout(title='Respiration Rate — Last 14 Days',
        xaxis_title='Date', yaxis_title='Breaths/min',
        template='plotly_white', height=400)
    fig.show()
    print(f"Avg waking respiration : {df_resp['wake_brpm'].mean():.1f} brpm")
    print(f"Avg sleep respiration  : {df_resp['sleep_brpm'].mean():.1f} brpm")
else:
    print("No respiration data returned.")


## 11 · Blood Oxygen (SpO₂) — Last 14 Days

In [ ]:
spo2_rows = []
for i in range(14):
    d = (TODAY - timedelta(days=i)).isoformat()
    try:
        data = api.get_spo2_data(d)
        if data and data.get('spO2HourlyAverages'):
            for entry in data['spO2HourlyAverages']:
                if entry.get('value'):
                    spo2_rows.append({
                        'time':  pd.to_datetime(entry['startGMT']),
                        'spo2':  entry['value']
                    })
    except Exception:
        pass

df_spo2 = pd.DataFrame(spo2_rows).sort_values('time')

if not df_spo2.empty:
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df_spo2['time'], y=df_spo2['spo2'],
        mode='lines', name='SpO₂',
        line=dict(color='#e74c3c', width=1),
        fill='tozeroy', fillcolor='rgba(231,76,60,0.1)'))
    fig.add_hline(y=95, line_dash='dash', line_color='orange',
        annotation_text='95% threshold')
    fig.update_layout(title='Blood Oxygen (SpO₂) — Last 14 Days',
        xaxis_title='Time', yaxis_title='SpO₂ %',
        yaxis=dict(range=[85, 100]),
        template='plotly_white', height=400)
    fig.show()
    print(f"Avg SpO₂  : {df_spo2['spo2'].mean():.1f}%")
    print(f"Min SpO₂  : {df_spo2['spo2'].min():.1f}%")
else:
    print("No SpO₂ data (requires supported Garmin device).")


## 12 · Health Snapshot (Today)

In [ ]:
from IPython.display import display, HTML

today_str = TODAY.isoformat()
snapshot = {}

try:
    hr = api.get_heart_rates(today_str)
    if hr:
        snapshot['Resting HR'] = f"{hr.get('restingHeartRate', 'N/A')} bpm"
except Exception: pass

try:
    steps = api.get_steps_data(today_str)
    if steps:
        snapshot['Steps Today'] = f"{sum(s.get('steps',0) for s in steps):,}"
except Exception: pass

try:
    sl = api.get_sleep_data(today_str)
    if sl and sl.get('dailySleepDTO'):
        secs = sl['dailySleepDTO'].get('sleepTimeSeconds') or 0
        snapshot['Last Night Sleep'] = f"{secs/3600:.2f} h"
        score = (sl['dailySleepDTO'].get('sleepScores') or {}).get('overall',{}).get('value')
        if score:
            snapshot['Sleep Score'] = str(score)
except Exception: pass

try:
    hrv = api.get_hrv_data(today_str)
    if hrv and hrv.get('hrvSummary'):
        snapshot['HRV (last night)'] = f"{hrv['hrvSummary'].get('lastNight','N/A')} ms"
        snapshot['HRV Status'] = hrv['hrvSummary'].get('status','N/A')
except Exception: pass

try:
    stress = api.get_stress_data(today_str)
    if stress:
        snapshot['Avg Stress'] = f"{stress.get('avgStressLevel','N/A')}/100"
except Exception: pass

table = "<h3 style='font-family:sans-serif'>Today's Health Snapshot — " + today_str + "</h3>"
table += "<table style='border-collapse:collapse;font-size:15px;font-family:sans-serif'>"
table += "<tr><th style='text-align:left;padding:8px 16px;background:#2c3e50;color:white'>Metric</th>"
table += "<th style='padding:8px 16px;background:#2c3e50;color:white'>Value</th></tr>"
colors = ['#ecf0f1', '#ffffff']
for idx, (k, v) in enumerate(snapshot.items()):
    bg = colors[idx % 2]
    table += f"<tr><td style='padding:7px 16px;background:{bg}'>{k}</td>"
    table += f"<td style='padding:7px 16px;background:{bg};font-weight:bold'>{v}</td></tr>"
table += "</table>"

display(HTML(table))
